# NOA System Testing & QA Notebook

This notebook provides comprehensive testing and quality assurance for the NOA (Neural Operating Architecture) system.

## Test Coverage
- **API Health & Connectivity** - Verify the API server is running and responsive
- **Memory CRUD Operations** - Create, read, update, list memories
- **Models Endpoint** - Test model management API
- **Database Integrity** - Verify data persistence and consistency
- **Performance Benchmarks** - Response time measurements
- **Error Handling** - Validate proper error responses
- **Data Validation** - Type and schema verification

## Prerequisites
- NOA API server running on `http://localhost:8080`
- Database initialized via `noa init`

## 1. Import Testing Libraries

Import the necessary libraries for HTTP requests, testing, timing, and data validation.

In [1]:
import requests
import json
import time
import uuid
from datetime import datetime
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from contextlib import contextmanager

# Test configuration
API_BASE_URL = "http://localhost:8080"
API_V1_URL = f"{API_BASE_URL}/api/v1"

# Test result tracking
@dataclass
class TestResult:
    name: str
    passed: bool
    duration_ms: float
    message: str = ""
    
test_results: List[TestResult] = []

def log_test(name: str, passed: bool, duration_ms: float, message: str = ""):
    """Log a test result with pass/fail indicator"""
    result = TestResult(name, passed, duration_ms, message)
    test_results.append(result)
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status} | {name} ({duration_ms:.2f}ms) {message}")
    return passed

@contextmanager
def timed_test(name: str):
    """Context manager for timing tests"""
    start = time.perf_counter()
    result = {"passed": False, "message": ""}
    try:
        yield result
    finally:
        duration = (time.perf_counter() - start) * 1000
        log_test(name, result["passed"], duration, result.get("message", ""))

print("✅ Testing libraries loaded successfully")

✅ Testing libraries loaded successfully


## 2. Set Up Test Fixtures

Create reusable test data and helper functions for consistent testing.

In [2]:
# Test fixtures - sample data for testing
class TestFixtures:
    """Reusable test data and helpers"""
    
    # Valid memory types (from MemoryType enum)
    VALID_MEMORY_TYPES = ["interaction", "decision", "learning", "artifact"]
    
    # Sample memories for testing
    SAMPLE_MEMORIES = [
        {
            "type": "decision",
            "content": "Test decision memory for QA validation",
            "tags": ["test", "qa", "automated"]
        },
        {
            "type": "learning",
            "content": "Learned new testing patterns for NOA system",
            "tags": ["test", "learning"]
        },
        {
            "type": "interaction",
            "content": "User interaction record for testing purposes",
            "tags": ["test", "interaction"]
        }
    ]
    
    # Invalid test cases for error handling
    INVALID_MEMORIES = [
        {"type": "invalid_type", "content": "Should fail validation"},
        {"content": "Missing type field"},
        {"type": "decision"},  # Missing content
    ]
    
    @staticmethod
    def generate_unique_memory(memory_type: str = "decision") -> Dict:
        """Generate a unique memory for testing"""
        return {
            "type": memory_type,
            "content": f"Test memory created at {datetime.now().isoformat()} - {uuid.uuid4().hex[:8]}",
            "tags": ["test", "generated", datetime.now().strftime("%Y%m%d")]
        }

# API helper functions
class NoaApiClient:
    """Helper class for NOA API interactions"""
    
    def __init__(self, base_url: str = API_V1_URL):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
    
    def get(self, endpoint: str, **kwargs) -> requests.Response:
        return self.session.get(f"{self.base_url}/{endpoint}", **kwargs)
    
    def post(self, endpoint: str, data: Dict, **kwargs) -> requests.Response:
        return self.session.post(f"{self.base_url}/{endpoint}", json=data, **kwargs)
    
    def delete(self, endpoint: str, **kwargs) -> requests.Response:
        return self.session.delete(f"{self.base_url}/{endpoint}", **kwargs)
    
    # Specific API methods
    def health_check(self) -> requests.Response:
        return self.session.get(f"{API_BASE_URL}/health")
    
    def list_memories(self, limit: int = 20, offset: int = 0) -> requests.Response:
        return self.get(f"memories?limit={limit}&offset={offset}")
    
    def create_memory(self, memory: Dict) -> requests.Response:
        return self.post("memories", memory)
    
    def get_memory(self, memory_id: str) -> requests.Response:
        return self.get(f"memories/{memory_id}")
    
    def list_models(self) -> requests.Response:
        return self.get("models")

# Initialize clients
fixtures = TestFixtures()
api = NoaApiClient()

print("✅ Test fixtures and API client initialized")

✅ Test fixtures and API client initialized


## 3. API Health & Connectivity Tests

Verify the NOA API server is running and responsive.

In [3]:
print("=" * 60)
print("API HEALTH & CONNECTIVITY TESTS")
print("=" * 60)

# Test 3.1: Health endpoint returns 200
with timed_test("Health endpoint returns 200") as result:
    try:
        response = api.health_check()
        result["passed"] = response.status_code == 200
        result["message"] = f"Status: {response.status_code}"
    except requests.exceptions.ConnectionError:
        result["message"] = "Connection refused - is the API server running?"

# Test 3.2: Health response contains required fields
with timed_test("Health response has required fields") as result:
    try:
        response = api.health_check()
        data = response.json()
        required_fields = ["status", "version", "uptime_secs", "components", "timestamp"]
        missing = [f for f in required_fields if f not in data]
        result["passed"] = len(missing) == 0
        result["message"] = f"Missing: {missing}" if missing else f"Version: {data.get('version')}"
    except Exception as e:
        result["message"] = str(e)

# Test 3.3: Database component is healthy
with timed_test("Database component healthy") as result:
    try:
        response = api.health_check()
        data = response.json()
        db_status = data.get("components", {}).get("database", {}).get("status")
        result["passed"] = db_status == "healthy"
        result["message"] = f"DB Status: {db_status}"
    except Exception as e:
        result["message"] = str(e)

# Test 3.4: API response time under 100ms
with timed_test("Health endpoint < 100ms response") as result:
    try:
        start = time.perf_counter()
        response = api.health_check()
        elapsed_ms = (time.perf_counter() - start) * 1000
        result["passed"] = elapsed_ms < 100
        result["message"] = f"Response time: {elapsed_ms:.2f}ms"
    except Exception as e:
        result["message"] = str(e)

print()

API HEALTH & CONNECTIVITY TESTS
✅ PASS | Health endpoint returns 200 (175.05ms) Status: 200
✅ PASS | Health response has required fields (3.84ms) Version: 0.1.0
✅ PASS | Database component healthy (2.97ms) DB Status: healthy
✅ PASS | Health endpoint < 100ms response (3.17ms) Response time: 3.16ms



## 4. Memory CRUD Operations

Test Create, Read, Update, and List operations for the Memory API.

In [4]:
print("=" * 60)
print("MEMORY CRUD OPERATIONS")
print("=" * 60)

created_memory_ids = []  # Track created memories for cleanup

# Test 4.1: Create memory with valid data
with timed_test("Create memory - valid data") as result:
    try:
        memory = fixtures.generate_unique_memory("decision")
        response = api.create_memory(memory)
        result["passed"] = response.status_code == 200
        if response.status_code == 200:
            data = response.json()
            if "id" in data:
                created_memory_ids.append(data["id"])
                result["message"] = f"Created ID: {data['id'][:8]}..."
        else:
            result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 4.2: Create memory for each valid type
for mem_type in fixtures.VALID_MEMORY_TYPES:
    with timed_test(f"Create memory - type '{mem_type}'") as result:
        try:
            memory = fixtures.generate_unique_memory(mem_type)
            response = api.create_memory(memory)
            result["passed"] = response.status_code == 200
            if response.status_code == 200:
                data = response.json()
                if "id" in data:
                    created_memory_ids.append(data["id"])
            result["message"] = f"Status: {response.status_code}"
        except Exception as e:
            result["message"] = str(e)

# Test 4.3: List memories returns array
with timed_test("List memories - returns array") as result:
    try:
        response = api.list_memories()
        data = response.json()
        result["passed"] = (
            response.status_code == 200 and 
            "memories" in data and 
            isinstance(data["memories"], list)
        )
        result["message"] = f"Count: {len(data.get('memories', []))}"
    except Exception as e:
        result["message"] = str(e)

# Test 4.4: List memories with pagination
with timed_test("List memories - pagination") as result:
    try:
        response = api.list_memories(limit=5, offset=0)
        data = response.json()
        result["passed"] = (
            response.status_code == 200 and
            "total" in data and
            "offset" in data and
            "limit" in data
        )
        result["message"] = f"Total: {data.get('total')}, Limit: {data.get('limit')}"
    except Exception as e:
        result["message"] = str(e)

# Test 4.5: Get memory by ID
with timed_test("Get memory by ID") as result:
    try:
        if created_memory_ids:
            mem_id = created_memory_ids[0]
            response = api.get_memory(mem_id)
            result["passed"] = response.status_code == 200
            result["message"] = f"Retrieved ID: {mem_id[:8]}..."
        else:
            result["message"] = "No memories created to retrieve"
    except Exception as e:
        result["message"] = str(e)

# Test 4.6: Get non-existent memory returns 404
with timed_test("Get non-existent memory - 404") as result:
    try:
        fake_id = str(uuid.uuid4())
        response = api.get_memory(fake_id)
        result["passed"] = response.status_code == 404
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

print(f"\n📊 Created {len(created_memory_ids)} test memories")

MEMORY CRUD OPERATIONS
✅ PASS | Create memory - valid data (21.26ms) Created ID: 59a3aaf2...
✅ PASS | Create memory - type 'interaction' (8.46ms) Status: 200
✅ PASS | Create memory - type 'decision' (9.40ms) Status: 200
✅ PASS | Create memory - type 'learning' (16.75ms) Status: 200
✅ PASS | Create memory - type 'artifact' (6.08ms) Status: 200
✅ PASS | List memories - returns array (6.57ms) Count: 6
✅ PASS | List memories - pagination (8.85ms) Total: 6, Limit: 5
✅ PASS | Get memory by ID (7.75ms) Retrieved ID: 59a3aaf2...
✅ PASS | Get non-existent memory - 404 (10.69ms) Status: 404

📊 Created 5 test memories


## 5. Validate Data Types and Shapes

Assert correct data types, field presence, and response structure.

## 6. Error Handling Validation

Test that invalid inputs produce appropriate error responses.

In [5]:
print("=" * 60)
print("DATA TYPE & SCHEMA VALIDATION")
print("=" * 60)

# Test 5.1: Memory response has correct field types
with timed_test("Memory response - field types") as result:
    try:
        response = api.list_memories(limit=1)
        data = response.json()
        memories = data.get("memories", [])
        
        if memories:
            mem = memories[0]
            type_checks = {
                "id": isinstance(mem.get("id"), str),
                "type": isinstance(mem.get("type"), str),
                "content": isinstance(mem.get("content"), str),
                "created_at": isinstance(mem.get("created_at"), str),
                "updated_at": isinstance(mem.get("updated_at"), str),
                "tags": isinstance(mem.get("tags"), (list, type(None))),
            }
            failed = [k for k, v in type_checks.items() if not v]
            result["passed"] = len(failed) == 0
            result["message"] = f"Failed: {failed}" if failed else "All types correct"
        else:
            result["passed"] = True
            result["message"] = "No memories to validate"
    except Exception as e:
        result["message"] = str(e)

# Test 5.2: Memory type is one of valid enum values
with timed_test("Memory type - valid enum") as result:
    try:
        response = api.list_memories()
        data = response.json()
        memories = data.get("memories", [])
        
        invalid_types = [m.get("type") for m in memories 
                        if m.get("type") not in fixtures.VALID_MEMORY_TYPES]
        result["passed"] = len(invalid_types) == 0
        result["message"] = f"Invalid: {invalid_types}" if invalid_types else f"Checked {len(memories)} memories"
    except Exception as e:
        result["message"] = str(e)

# Test 5.3: UUID format validation
with timed_test("Memory ID - UUID format") as result:
    try:
        response = api.list_memories()
        data = response.json()
        memories = data.get("memories", [])
        
        invalid_ids = []
        for m in memories:
            try:
                uuid.UUID(m.get("id", ""))
            except ValueError:
                invalid_ids.append(m.get("id"))
        
        result["passed"] = len(invalid_ids) == 0
        result["message"] = f"Invalid IDs: {len(invalid_ids)}" if invalid_ids else f"Validated {len(memories)} IDs"
    except Exception as e:
        result["message"] = str(e)

# Test 5.4: Timestamp format (ISO 8601)
with timed_test("Timestamps - ISO 8601 format") as result:
    try:
        response = api.list_memories(limit=1)
        data = response.json()
        memories = data.get("memories", [])
        
        if memories:
            mem = memories[0]
            created_at = mem.get("created_at", "")
            # Check if it contains expected ISO format components
            result["passed"] = "T" in created_at and ("+" in created_at or "Z" in created_at)
            result["message"] = f"Timestamp: {created_at[:25]}..."
        else:
            result["passed"] = True
            result["message"] = "No memories to validate"
    except Exception as e:
        result["message"] = str(e)

# Test 5.5: Checksum field is present and valid hex
with timed_test("Memory checksum - valid hex") as result:
    try:
        response = api.list_memories(limit=1)
        data = response.json()
        memories = data.get("memories", [])
        
        if memories:
            checksum = memories[0].get("checksum", "")
            # SHA256 produces 64 hex characters
            is_valid_hex = all(c in "0123456789abcdef" for c in checksum.lower())
            result["passed"] = len(checksum) == 64 and is_valid_hex
            result["message"] = f"Checksum: {checksum[:16]}... (len={len(checksum)})"
        else:
            result["passed"] = True
            result["message"] = "No memories to validate"
    except Exception as e:
        result["message"] = str(e)

print()

DATA TYPE & SCHEMA VALIDATION
✅ PASS | Memory response - field types (12.91ms) All types correct
✅ PASS | Memory type - valid enum (5.89ms) Checked 6 memories
✅ PASS | Memory ID - UUID format (7.33ms) Validated 6 IDs
✅ PASS | Timestamps - ISO 8601 format (15.30ms) Timestamp: 2025-12-11T14:33:05.16942...
✅ PASS | Memory checksum - valid hex (9.87ms) Checksum: 010450a276d39e7c... (len=64)



In [6]:
print("=" * 60)
print("ERROR HANDLING VALIDATION")
print("=" * 60)

# Test 6.1: Invalid memory type returns 400
with timed_test("Invalid memory type - 400 error") as result:
    try:
        invalid_memory = {"type": "invalid_type", "content": "Test content"}
        response = api.create_memory(invalid_memory)
        result["passed"] = response.status_code == 400
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 6.2: Missing required field returns 422
with timed_test("Missing content field - 422 error") as result:
    try:
        invalid_memory = {"type": "decision"}  # Missing 'content'
        response = api.create_memory(invalid_memory)
        result["passed"] = response.status_code == 422
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 6.3: Missing type field returns 422
with timed_test("Missing type field - 422 error") as result:
    try:
        invalid_memory = {"content": "Test content"}  # Missing 'type'
        response = api.create_memory(invalid_memory)
        result["passed"] = response.status_code == 422
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 6.4: Invalid UUID format returns 400
with timed_test("Invalid UUID format - 400 error") as result:
    try:
        response = api.get_memory("not-a-valid-uuid")
        result["passed"] = response.status_code == 400
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 6.5: Error response has correct structure
with timed_test("Error response structure") as result:
    try:
        response = api.create_memory({"type": "invalid"})
        data = response.json()
        has_error = "error" in data
        has_code = "code" in data
        result["passed"] = has_error and has_code
        result["message"] = f"Has error: {has_error}, Has code: {has_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 6.6: Empty JSON body returns 422
with timed_test("Empty request body - 422 error") as result:
    try:
        response = api.create_memory({})
        result["passed"] = response.status_code == 422
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

print()

ERROR HANDLING VALIDATION
✅ PASS | Invalid memory type - 400 error (3.70ms) Status: 400
✅ PASS | Missing content field - 422 error (1.95ms) Status: 422
✅ PASS | Missing type field - 422 error (1.34ms) Status: 422
✅ PASS | Invalid UUID format - 400 error (8.12ms) Status: 400
❌ FAIL | Error response structure (1.46ms) Expecting value: line 1 column 1 (char 0)
✅ PASS | Empty request body - 422 error (1.30ms) Status: 422



## 7. Models Endpoint Testing

Test the AI models management API.

In [7]:
print("=" * 60)
print("MODELS ENDPOINT TESTING")
print("=" * 60)

# Test 7.1: Models endpoint returns 200
with timed_test("Models endpoint - 200 OK") as result:
    try:
        response = api.list_models()
        result["passed"] = response.status_code == 200
        result["message"] = f"Status: {response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 7.2: Models response has correct structure
with timed_test("Models response structure") as result:
    try:
        response = api.list_models()
        data = response.json()
        result["passed"] = "models" in data and isinstance(data["models"], list)
        result["message"] = f"Models count: {len(data.get('models', []))}"
    except Exception as e:
        result["message"] = str(e)

# Test 7.3: Models response time under 50ms
with timed_test("Models endpoint < 50ms") as result:
    try:
        start = time.perf_counter()
        response = api.list_models()
        elapsed_ms = (time.perf_counter() - start) * 1000
        result["passed"] = elapsed_ms < 50 and response.status_code == 200
        result["message"] = f"Response time: {elapsed_ms:.2f}ms"
    except Exception as e:
        result["message"] = str(e)

print()

MODELS ENDPOINT TESTING
✅ PASS | Models endpoint - 200 OK (9.80ms) Status: 200
✅ PASS | Models response structure (4.12ms) Models count: 0
✅ PASS | Models endpoint < 50ms (5.78ms) Response time: 5.77ms



## 8. Performance Benchmarks

Measure response times and throughput.

In [8]:
print("=" * 60)
print("PERFORMANCE BENCHMARKS")
print("=" * 60)

def benchmark_endpoint(name: str, request_func, iterations: int = 10):
    """Run multiple requests and measure timing statistics"""
    times = []
    errors = 0
    
    for _ in range(iterations):
        try:
            start = time.perf_counter()
            response = request_func()
            elapsed = (time.perf_counter() - start) * 1000
            if response.status_code == 200:
                times.append(elapsed)
            else:
                errors += 1
        except Exception:
            errors += 1
    
    if times:
        return {
            "name": name,
            "min_ms": min(times),
            "max_ms": max(times),
            "avg_ms": sum(times) / len(times),
            "iterations": iterations,
            "errors": errors
        }
    return {"name": name, "error": "All requests failed"}

# Benchmark 8.1: Health endpoint
print("\n📊 Benchmarking endpoints (10 iterations each)...\n")

health_bench = benchmark_endpoint("Health", api.health_check)
print(f"  Health:   avg={health_bench['avg_ms']:.2f}ms, min={health_bench['min_ms']:.2f}ms, max={health_bench['max_ms']:.2f}ms")

# Benchmark 8.2: Models endpoint
models_bench = benchmark_endpoint("Models", api.list_models)
print(f"  Models:   avg={models_bench['avg_ms']:.2f}ms, min={models_bench['min_ms']:.2f}ms, max={models_bench['max_ms']:.2f}ms")

# Benchmark 8.3: Memories list endpoint
memories_bench = benchmark_endpoint("Memories", lambda: api.list_memories(limit=10))
print(f"  Memories: avg={memories_bench['avg_ms']:.2f}ms, min={memories_bench['min_ms']:.2f}ms, max={memories_bench['max_ms']:.2f}ms")

# Benchmark 8.4: Memory creation throughput
print("\n📊 Memory creation throughput (5 iterations)...")
create_times = []
for i in range(5):
    memory = fixtures.generate_unique_memory()
    start = time.perf_counter()
    response = api.create_memory(memory)
    elapsed = (time.perf_counter() - start) * 1000
    if response.status_code == 200:
        create_times.append(elapsed)
        data = response.json()
        if "id" in data:
            created_memory_ids.append(data["id"])

if create_times:
    print(f"  Create:   avg={sum(create_times)/len(create_times):.2f}ms, min={min(create_times):.2f}ms, max={max(create_times):.2f}ms")

# Performance assertions
with timed_test("Avg health response < 20ms") as result:
    result["passed"] = health_bench.get("avg_ms", 999) < 20
    result["message"] = f"Avg: {health_bench.get('avg_ms', 'N/A'):.2f}ms"

with timed_test("Avg memories response < 50ms") as result:
    result["passed"] = memories_bench.get("avg_ms", 999) < 50
    result["message"] = f"Avg: {memories_bench.get('avg_ms', 'N/A'):.2f}ms"

print()

PERFORMANCE BENCHMARKS

📊 Benchmarking endpoints (10 iterations each)...

  Health:   avg=3.30ms, min=2.55ms, max=4.82ms
  Models:   avg=5.95ms, min=4.46ms, max=11.13ms
  Memories: avg=10.45ms, min=7.84ms, max=16.45ms

📊 Memory creation throughput (5 iterations)...
  Create:   avg=9.57ms, min=8.13ms, max=10.94ms
✅ PASS | Avg health response < 20ms (0.01ms) Avg: 3.30ms
✅ PASS | Avg memories response < 50ms (0.03ms) Avg: 10.45ms



## 9. Data Integrity Checks

Verify data persistence and consistency.

In [9]:
print("=" * 60)
print("DATA INTEGRITY CHECKS")
print("=" * 60)

# Test 9.1: Created memories persist
with timed_test("Created memories persist") as result:
    try:
        if created_memory_ids:
            mem_id = created_memory_ids[-1]  # Get most recent
            response = api.get_memory(mem_id)
            result["passed"] = response.status_code == 200
            result["message"] = f"Retrieved persisted memory: {mem_id[:8]}..."
        else:
            result["passed"] = False
            result["message"] = "No memories created"
    except Exception as e:
        result["message"] = str(e)

# Test 9.2: Memory content matches what was stored
with timed_test("Memory content integrity") as result:
    try:
        # Create a memory with known content
        test_content = f"Integrity test {uuid.uuid4().hex[:8]}"
        memory = {
            "type": "learning",
            "content": test_content,
            "tags": ["integrity-test"]
        }
        
        create_response = api.create_memory(memory)
        if create_response.status_code == 200:
            data = create_response.json()
            mem_id = data.get("id")
            created_memory_ids.append(mem_id)
            
            # Retrieve and verify
            get_response = api.get_memory(mem_id)
            if get_response.status_code == 200:
                retrieved = get_response.json()
                result["passed"] = retrieved.get("content") == test_content
                result["message"] = "Content matches" if result["passed"] else "Content mismatch!"
            else:
                result["message"] = f"Get failed: {get_response.status_code}"
        else:
            result["message"] = f"Create failed: {create_response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 9.3: Tags persist correctly
with timed_test("Tags persist correctly") as result:
    try:
        test_tags = ["tag1", "tag2", "special-tag-123"]
        memory = {
            "type": "artifact",
            "content": "Tags persistence test",
            "tags": test_tags
        }
        
        create_response = api.create_memory(memory)
        if create_response.status_code == 200:
            data = create_response.json()
            mem_id = data.get("id")
            created_memory_ids.append(mem_id)
            
            get_response = api.get_memory(mem_id)
            if get_response.status_code == 200:
                retrieved = get_response.json()
                stored_tags = retrieved.get("tags", [])
                result["passed"] = set(stored_tags) == set(test_tags)
                result["message"] = f"Tags: {stored_tags}"
            else:
                result["message"] = f"Get failed: {get_response.status_code}"
        else:
            result["message"] = f"Create failed: {create_response.status_code}"
    except Exception as e:
        result["message"] = str(e)

# Test 9.4: Total count increases after creation
with timed_test("Total count increases after creation") as result:
    try:
        # Get initial count
        response1 = api.list_memories()
        initial_count = response1.json().get("total", 0)
        
        # Create new memory
        memory = fixtures.generate_unique_memory()
        create_response = api.create_memory(memory)
        if create_response.status_code == 200:
            data = create_response.json()
            created_memory_ids.append(data.get("id"))
        
        # Get new count
        response2 = api.list_memories()
        new_count = response2.json().get("total", 0)
        
        result["passed"] = new_count == initial_count + 1
        result["message"] = f"Count: {initial_count} → {new_count}"
    except Exception as e:
        result["message"] = str(e)

print(f"\n📊 Total test memories created: {len(created_memory_ids)}")

DATA INTEGRITY CHECKS
✅ PASS | Created memories persist (15.39ms) Retrieved persisted memory: 2f2a9898...
✅ PASS | Memory content integrity (50.29ms) Content matches
✅ PASS | Tags persist correctly (27.84ms) Tags: ['tag2', 'special-tag-123', 'tag1']
✅ PASS | Total count increases after creation (31.44ms) Count: 13 → 14

📊 Total test memories created: 13


## 10. Test Summary & Report

Generate a comprehensive test report with pass/fail statistics.

In [10]:
print("=" * 60)
print("TEST SUMMARY & REPORT")
print("=" * 60)

# Calculate statistics
total_tests = len(test_results)
passed_tests = sum(1 for r in test_results if r.passed)
failed_tests = total_tests - passed_tests
pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

# Calculate timing stats
all_times = [r.duration_ms for r in test_results]
avg_time = sum(all_times) / len(all_times) if all_times else 0
total_time = sum(all_times)

print(f"""
╔══════════════════════════════════════════════════════════╗
║                    NOA QA TEST REPORT                    ║
╠══════════════════════════════════════════════════════════╣
║  Total Tests:     {total_tests:>4}                                   ║
║  Passed:          {passed_tests:>4}  ✅                                ║
║  Failed:          {failed_tests:>4}  {'❌' if failed_tests > 0 else '  '}                                ║
║  Pass Rate:       {pass_rate:>5.1f}%                                 ║
╠══════════════════════════════════════════════════════════╣
║  Total Time:      {total_time:>7.2f}ms                             ║
║  Avg per Test:    {avg_time:>7.2f}ms                             ║
╚══════════════════════════════════════════════════════════╝
""")

# List failed tests if any
if failed_tests > 0:
    print("\n❌ FAILED TESTS:")
    print("-" * 60)
    for r in test_results:
        if not r.passed:
            print(f"  • {r.name}: {r.message}")
    print()

# Summary by category
print("\n📊 RESULTS BY CATEGORY:")
print("-" * 60)

categories = {
    "Health": [r for r in test_results if "health" in r.name.lower() or "Health" in r.name],
    "Memory CRUD": [r for r in test_results if "memory" in r.name.lower() or "Memory" in r.name],
    "Models": [r for r in test_results if "model" in r.name.lower()],
    "Error Handling": [r for r in test_results if "error" in r.name.lower() or "invalid" in r.name.lower() or "missing" in r.name.lower()],
    "Data Types": [r for r in test_results if "type" in r.name.lower() or "format" in r.name.lower() or "UUID" in r.name],
    "Performance": [r for r in test_results if "ms" in r.name.lower() or "Avg" in r.name],
    "Integrity": [r for r in test_results if "persist" in r.name.lower() or "integrity" in r.name.lower() or "count" in r.name.lower()],
}

for category, tests in categories.items():
    if tests:
        cat_passed = sum(1 for t in tests if t.passed)
        status = "✅" if cat_passed == len(tests) else "⚠️" if cat_passed > 0 else "❌"
        print(f"  {status} {category}: {cat_passed}/{len(tests)} passed")

# Final verdict
print("\n" + "=" * 60)
if pass_rate == 100:
    print("🎉 ALL TESTS PASSED - NOA SYSTEM IS HEALTHY!")
elif pass_rate >= 80:
    print("⚠️  MOSTLY PASSING - Review failed tests")
else:
    print("❌ CRITICAL ISSUES DETECTED - Immediate attention required")
print("=" * 60)

TEST SUMMARY & REPORT

╔══════════════════════════════════════════════════════════╗
║                    NOA QA TEST REPORT                    ║
╠══════════════════════════════════════════════════════════╣
║  Total Tests:       33                                   ║
║  Passed:            32  ✅                                ║
║  Failed:             1  ❌                                ║
║  Pass Rate:        97.0%                                 ║
╠══════════════════════════════════════════════════════════╣
║  Total Time:       494.69ms                             ║
║  Avg per Test:      14.99ms                             ║
╚══════════════════════════════════════════════════════════╝


❌ FAILED TESTS:
------------------------------------------------------------
  • Error response structure: Expecting value: line 1 column 1 (char 0)


📊 RESULTS BY CATEGORY:
------------------------------------------------------------
  ✅ Health: 5/5 passed
  ✅ Memory CRUD: 13/13 passed
  ✅ Models: 3/3 pa